# Evaluate TheSystem
Compare Probe-256, LoRA and original Qwen on the same five benchmarks, then check `<FAST>` routing and generation preservation. No training is performed.


## Before you start
Choose **Runtime → Change runtime type** and select a GPU with native BF16 support, such as **L4** or **A100**. Use Python 3.12 or 3.13. Have a Hugging Face read token for both model repositories and `TheSystem-Data` ready.


## 1. Prepare runtime
Run this cell and enter your read token. If setup asks for a restart, choose **Runtime → Restart session**. When Colab reconnects, click **Run all**. If you see **READY**, continue to step 2.


In [ ]:
# @title Prepare runtime
RUNTIME_READY = OUTPUT_READY = RESOURCES_READY = False
import getpass, hashlib, json, os, sys
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

PREPARE = {'repo_id': 'Lbuk/TheSystem-9B-Probe-256', 'revision': 'd66b3452f760f9efa5ba8ffecfb6b1759a94f9ff', 'sha256': 'f2a68dee15d95d2a9bda090e69c50232b1fb245da6e68b49ea4dd7f9d938f1b5'}
if not globals().get("RESTART_PENDING", False):
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise RuntimeError("Use Python 3.12 or 3.13: Colab Runtime → Change runtime type → the latest runtime version.")
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face read token (hidden; Enter only for public access): ").strip()
    print("Downloading setup…", flush=True)
    url = f"https://huggingface.co/{PREPARE['repo_id']}/resolve/{PREPARE['revision']}/prepare_evaluation.py"
    headers = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"} if os.environ["HF_TOKEN"] else {}
    try:
        with urlopen(Request(url, headers=headers), timeout=30) as response:
            helper = response.read()
    except HTTPError as error:
        raise RuntimeError("Setup download failed. For HTTP 401/403, rerun this cell and paste a valid read token for this private repository. "
                           f"HTTP status: {error.code}. Your browser login is separate from Colab.") from None
    except (URLError, TimeoutError) as error:
        raise RuntimeError("Setup helper download failed or timed out. Check the connection and rerun Prepare runtime.") from None
    if hashlib.sha256(helper).hexdigest() != PREPARE["sha256"]:
        raise RuntimeError("Setup helper checksum mismatch. Stop and obtain a fresh release notebook.")
    _setup = {"__name__": "thesystem_evaluation_setup"}
    exec(compile(helper, "prepare_evaluation.py", "exec"), _setup)
    visible_progress = _setup["visible_progress"]
    RUNTIME_READY = _setup["prepare_runtime"]()
    RESTART_PENDING = not RUNTIME_READY

if RESTART_PENDING:
    from IPython.display import Markdown, clear_output, display
    clear_output(wait=True)
    display(Markdown("**Setup saved. Choose Runtime → Restart session.**  \nAfter Colab reconnects, click **Run all**."))


## 2. Choose run and save location
Leave `SMOKE = False` for the full evaluation; `True` runs a small mechanics check in a separate `-smoke` folder. Results default to **Colab local storage**; no Drive access is requested unless you select `USE_DRIVE = True`.

Local Colab files disappear when the runtime is deleted. Download your results in step 5 before confirming shutdown; optional Drive storage preserves progress across sessions. All model weights and datasets come from Hugging Face.


In [ ]:
# @title Choose run and save location
RUN_NAME = "release-evaluation-rc4"  # @param {type:"string"}
SMOKE = False  # @param {type:"boolean"}
USE_DRIVE = False  # @param {type:"boolean"}

if not globals().get("RESTART_PENDING", False):
    if not globals().get("RUNTIME_READY", False):
        raise RuntimeError("Run Prepare runtime until it reports READY first.")
    OUTPUT_READY = RESOURCES_READY = False
    from datetime import datetime, timezone
    from thesystem.evaluation import evaluate
    if SMOKE:
        RUN_NAME += "-smoke"
    OUTPUT, CACHE, IN_COLAB, runtime = _setup["prepare_output"](RUN_NAME, USE_DRIVE)
    def run_job():
        return evaluate(RESOURCE_ROOT, OUTPUT, cache=CACHE, smoke=SMOKE)
    OUTPUT_READY = True
    print("Selected:", "QUICK CHECK — not benchmark scores" if SMOKE else "FULL EVALUATION — all 1,648 benchmark examples per model")


## 3. Download resources
Download and checksum-verify the exact model assets, tokenizer and datasets. Downloads stay on local runtime disk, while new results go to the output folder chosen above; the pinned Qwen backbone loads when evaluation starts.


In [ ]:
# @title Download resources
if not globals().get("RESTART_PENDING", False):
    if not globals().get("RUNTIME_READY", False) or not globals().get("OUTPUT_READY", False):
        raise RuntimeError("Complete Prepare runtime and Choose run and save location first.")
    RESOURCES_READY = False
    from thesystem.artifacts import resolve_release_resources
    with visible_progress("Download and verify release resources"):
        RESOURCE_ROOT = resolve_release_resources()
    print("Verified release resources:", RESOURCE_ROOT)
    RESOURCES_READY = True


## 4. Run evaluation and view results
Models run one at a time with live logs and heartbeats. The table below follows the website: Benchmark, `<FAST>` + LoRA, `<FAST>` + Probe-256, Bespoke-Nimble-9B and Jev 1.13.0; original Qwen, routing and generation details remain in the downloaded report.

Results and status are saved locally even after a handled failure. **This cell never shuts down the runtime.** After a failure, run step 5 to download partial results and logs.


In [ ]:
# @title Run evaluation and save results
if not globals().get("RESTART_PENDING", False):
    if not all(globals().get(name, False) for name in ("RUNTIME_READY", "OUTPUT_READY", "RESOURCES_READY")):
        raise RuntimeError("Complete steps 1–3 before running evaluation.")
    status = "failed"
    started = datetime.now(timezone.utc).isoformat()
    try:
        RESULT_ROOT = run_job()
        summary_path = Path(RESULT_ROOT) / "SUMMARY.json"
        summary = json.loads(summary_path.read_text())
        if not summary.get("complete"):
            raise RuntimeError("The workflow did not report complete; inspect saved logs before claiming success.")
        status = "completed"
        print("Completed. Results and logs are saved in:", RESULT_ROOT)
        print("Run type:", "SMOKE CHECK — not full results" if SMOKE else "FULL EVALUATION")
        print("Report:", Path(RESULT_ROOT) / "REPORT.html")
        print("Score table:", Path(RESULT_ROOT) / "comparison.csv")
        print("Summary:", summary_path)
    finally:
        # If this persistent write fails, do not disconnect the runtime.
        record = {"status": status, "started_utc": started,
                  "finished_utc": datetime.now(timezone.utc).isoformat(),
                  "run_name": RUN_NAME, "output": str(OUTPUT)}
        with (OUTPUT / "notebook-status.json").open("w") as handle:
            json.dump(record, handle, indent=2)
            handle.flush()
            os.fsync(handle.fileno())

    _setup["show_results"](OUTPUT)


## 5. Download results and finish
Click **Download results ZIP** and open it on your computer. Then tick the confirmation and click **Shut down runtime**. If the buttons are missing, use the [download instructions](https://github.com/bqpro1/TheSystem/blob/main/docs/REPRODUCIBILITY.md#evaluation-results-and-storage).


In [ ]:
# @title Download results and finish
if globals().get("RESTART_PENDING", False):
    print("Paused at step 1. Choose Runtime → Restart session, then Run all.")
else:
    if not globals().get("OUTPUT_READY", False):
        raise RuntimeError("Choose an output folder first.")
    finish_controls = _setup["finish_controls"](OUTPUT)


## Reading your results
The full run measures VitaminC (599), PAWS (250), SQuAD2 answerability (299), Aegis2 (250) and PubMedQA (250), plus the 4,000-request internal routing test, external routing checks and 45 writing prompts. Automatic accuracy counts routing errors; generation checks compare output tokens with original Qwen and report truncation.

Nimble/Jev scores are published references with different prompt wrappers; internal tests were inspected during development, and PubMedQA uses the publisher's subset rather than the official leaderboard split. See the [evaluation guide](https://github.com/bqpro1/TheSystem/blob/main/docs/EVALUATION.md) for interpretation; a quick check is not a full benchmark result.
